In [1]:
import numpy as np
import pandas as pd
from paraphrase_detection import data_shuffle_split, SBERTPairClassifier_model_selection, Train, TextPairDataset, log_metrics_and_model
import torch
from sentence_transformers import SentenceTransformer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import os
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
df = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair-class/train-00000-of-00001.parquet")
data = df.to_numpy()
data = data[:5000]

X = data[:, :2]
y = data[:, 2:3]

X_train, X_val, X_test, y_train, y_val, y_test = data_shuffle_split(X, y, 0.15, 0.12, 42)

In [3]:
seed = 42
model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [model.get_sentence_embedding_dimension()]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size #TODO batch size or number of batches
n_warmup_steps = steps * 0.1
n_freeze = 3

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier_model_selection(model, fc_layer_sizes, device, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

In [4]:
train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train), batch_size = batch_size, shuffle = True, num_workers = 4)
validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val), batch_size = batch_size, shuffle = True, num_workers = 4)
test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test), batch_size = batch_size, shuffle = True, num_workers = 4)

tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")
tokenized_train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)
tokenized_validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)
tokenized_test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)

In [13]:
seed = 42
model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [model.get_sentence_embedding_dimension()]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size #TODO batch size or number of batches
n_warmup_steps = steps * 0.1
n_freeze = 3

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier_model_selection(model, fc_layer_sizes, device, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

In [14]:
#Fixed sbert model
train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = train_loader,
              val_dataloader = validation_loader
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()

EPOCH: 0


2025-11-17 09:24:42.942 | INFO     | paraphrase_detection.train:run_training_loop:121 - Epoch 0: train loss = 0.5541775053382939
2025-11-17 09:24:42.942 | INFO     | paraphrase_detection.train:run_training_loop:122 - Epoch 0: validation loss = 0.40817790292203426
2025-11-17 09:24:42.942 | INFO     | paraphrase_detection.train:run_training_loop:123 - Epoch 0: train acc = 0.6657754010695187
2025-11-17 09:24:42.942 | INFO     | paraphrase_detection.train:run_training_loop:124 - Epoch 0: validation acc = 0.7901960784313725
2025-11-17 09:24:42.942 | INFO     | paraphrase_detection.train:run_training_loop:125 - Epoch 0: validation f1 = 0.7763258747965947


EPOCH: 1


2025-11-17 09:26:11.806 | INFO     | paraphrase_detection.train:run_training_loop:121 - Epoch 1: train loss = 0.4005055529439551
2025-11-17 09:26:11.807 | INFO     | paraphrase_detection.train:run_training_loop:122 - Epoch 1: validation loss = 0.3980018887668848
2025-11-17 09:26:11.807 | INFO     | paraphrase_detection.train:run_training_loop:123 - Epoch 1: train acc = 0.8045454545454546
2025-11-17 09:26:11.808 | INFO     | paraphrase_detection.train:run_training_loop:124 - Epoch 1: validation acc = 0.796078431372549
2025-11-17 09:26:11.809 | INFO     | paraphrase_detection.train:run_training_loop:125 - Epoch 1: validation f1 = 0.7794594594594595


EPOCH: 2


...

2025-11-17 09:39:34.334 | INFO     | paraphrase_detection.train:run_training_loop:121 - Epoch 7: train loss = 0.18225995648620474
2025-11-17 09:39:34.348 | INFO     | paraphrase_detection.train:run_training_loop:122 - Epoch 7: validation loss = 0.41536950785666704
2025-11-17 09:39:34.348 | INFO     | paraphrase_detection.train:run_training_loop:123 - Epoch 7: train acc = 0.9336898395721925
2025-11-17 09:39:34.350 | INFO     | paraphrase_detection.train:run_training_loop:124 - Epoch 7: validation acc = 0.8098039215686275
2025-11-17 09:39:34.350 | INFO     | paraphrase_detection.train:run_training_loop:125 - Epoch 7: validation f1 = 0.7996151898734176


EPOCH: 8


2025-11-17 09:45:08.653 | INFO     | paraphrase_detection.train:run_training_loop:121 - Epoch 8: train loss = 0.16634990211225983
2025-11-17 09:45:08.653 | INFO     | paraphrase_detection.train:run_training_loop:122 - Epoch 8: validation loss = 0.4178373469039798
2025-11-17 09:45:08.656 | INFO     | paraphrase_detection.train:run_training_loop:123 - Epoch 8: train acc = 0.9451871657754011
2025-11-17 09:45:08.657 | INFO     | paraphrase_detection.train:run_training_loop:124 - Epoch 8: validation acc = 0.8117647058823529
2025-11-17 09:45:08.659 | INFO     | paraphrase_detection.train:run_training_loop:125 - Epoch 8: validation f1 = 0.8042383046781287


EPOCH: 9


2025-11-17 09:50:30.562 | INFO     | paraphrase_detection.train:run_training_loop:121 - Epoch 9: train loss = 0.15631054774818257
2025-11-17 09:50:30.565 | INFO     | paraphrase_detection.train:run_training_loop:122 - Epoch 9: validation loss = 0.41927223559468985
2025-11-17 09:50:30.568 | INFO     | paraphrase_detection.train:run_training_loop:123 - Epoch 9: train acc = 0.9518716577540107
2025-11-17 09:50:30.570 | INFO     | paraphrase_detection.train:run_training_loop:124 - Epoch 9: validation acc = 0.8117647058823529
2025-11-17 09:50:30.572 | INFO     | paraphrase_detection.train:run_training_loop:125 - Epoch 9: validation f1 = 0.803921568627451


In [ ]:
log_metrics_and_model('SBERTv1', avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, \
                      epoch_val_acc, epoch_val_f1, best_params)

In [5]:
#Trainable Sbert model

seed = 42
model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [model.get_sentence_embedding_dimension()]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size #TODO batch size or number of batches
n_warmup_steps = steps * 0.1
n_freeze = 3

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier_model_selection(model, fc_layer_sizes, device, dropout_p,trainable=True)
optimizer = torch.optim.AdamW(model.parameters(), lr)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

In [6]:
#Trainable Sbert model

train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = tokenized_train_loader,
              val_dataloader = tokenized_validation_loader,
              sbert_trainable=True
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()

False
False
False
False
False
EPOCH: 0


2025-11-17 11:06:49.402 | INFO     | paraphrase_detection.train:run_training_loop:126 - Epoch 0: train loss = 0.5816728263838679
2025-11-17 11:06:49.404 | INFO     | paraphrase_detection.train:run_training_loop:127 - Epoch 0: validation loss = 0.4858977794647217
2025-11-17 11:06:49.405 | INFO     | paraphrase_detection.train:run_training_loop:128 - Epoch 0: train acc = 0.6754010695187166
2025-11-17 11:06:49.408 | INFO     | paraphrase_detection.train:run_training_loop:129 - Epoch 0: validation acc = 0.792156862745098
2025-11-17 11:06:49.409 | INFO     | paraphrase_detection.train:run_training_loop:130 - Epoch 0: validation f1 = 0.7838464614154339


EPOCH: 1


2025-11-17 11:19:43.721 | INFO     | paraphrase_detection.train:run_training_loop:126 - Epoch 1: train loss = 0.5071015999867365
2025-11-17 11:19:43.721 | INFO     | paraphrase_detection.train:run_training_loop:127 - Epoch 1: validation loss = 0.4782915282994509
2025-11-17 11:19:43.721 | INFO     | paraphrase_detection.train:run_training_loop:128 - Epoch 1: train acc = 0.7451871657754011
2025-11-17 11:19:43.721 | INFO     | paraphrase_detection.train:run_training_loop:129 - Epoch 1: validation acc = 0.7745098039215687
2025-11-17 11:19:43.721 | INFO     | paraphrase_detection.train:run_training_loop:130 - Epoch 1: validation f1 = 0.7705497807214869


EPOCH: 2


...

2025-11-17 12:38:07.926 | INFO     | paraphrase_detection.train:run_training_loop:126 - Epoch 7: train loss = 0.15757471467885706
2025-11-17 12:38:07.927 | INFO     | paraphrase_detection.train:run_training_loop:127 - Epoch 7: validation loss = 0.6495482660830021
2025-11-17 12:38:07.928 | INFO     | paraphrase_detection.train:run_training_loop:128 - Epoch 7: train acc = 0.9449197860962567
2025-11-17 12:38:07.931 | INFO     | paraphrase_detection.train:run_training_loop:129 - Epoch 7: validation acc = 0.7941176470588235
2025-11-17 12:38:07.933 | INFO     | paraphrase_detection.train:run_training_loop:130 - Epoch 7: validation f1 = 0.7873472613265877


EPOCH: 8


2025-11-17 12:50:30.224 | INFO     | paraphrase_detection.train:run_training_loop:126 - Epoch 8: train loss = 0.1009917438794405
2025-11-17 12:50:30.226 | INFO     | paraphrase_detection.train:run_training_loop:127 - Epoch 8: validation loss = 0.6581377107650042
2025-11-17 12:50:30.227 | INFO     | paraphrase_detection.train:run_training_loop:128 - Epoch 8: train acc = 0.9703208556149733
2025-11-17 12:50:30.227 | INFO     | paraphrase_detection.train:run_training_loop:129 - Epoch 8: validation acc = 0.8
2025-11-17 12:50:30.228 | INFO     | paraphrase_detection.train:run_training_loop:130 - Epoch 8: validation f1 = 0.7961598746081504


EPOCH: 9


2025-11-17 13:02:35.397 | INFO     | paraphrase_detection.train:run_training_loop:126 - Epoch 9: train loss = 0.06698548435591735
2025-11-17 13:02:35.398 | INFO     | paraphrase_detection.train:run_training_loop:127 - Epoch 9: validation loss = 0.6672219038009644
2025-11-17 13:02:35.398 | INFO     | paraphrase_detection.train:run_training_loop:128 - Epoch 9: train acc = 0.9820855614973262
2025-11-17 13:02:35.398 | INFO     | paraphrase_detection.train:run_training_loop:129 - Epoch 9: validation acc = 0.796078431372549
2025-11-17 13:02:35.398 | INFO     | paraphrase_detection.train:run_training_loop:130 - Epoch 9: validation f1 = 0.7916928490637174
